# Notebook 08: Explainability with Qwen-3

Loads the trained uncertainty model from Notebook 07, runs inference on five
representative test-set records, computes gradient saliency maps to identify
which ECG regions and leads drove each prediction, then feeds the structured
output to **Qwen3-0.6B** to produce a plain-language clinical interpretation.

**Cells run in order.** Requires a trained `results/uncertainty_model/` from Notebook 07.
Qwen3 is downloaded from HuggingFace on first run (~400 MB).

In [ ]:
import sys, os, json, warnings, time
sys.path.append('../')
# HuBERT must load from local cache -- set BEFORE importing transformers
os.environ['TRANSFORMERS_OFFLINE']          = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
warnings.filterwarnings('ignore')

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pandas as pd

from src.utils.config               import CFG
from src.models.uncertainty_head    import AleatoricWrapper
from src.inference.pipeline         import ECGInferencePipeline
from src.inference.loaders          import load_hubert_blocks, load_hubert_peft, load_leadwise
from src.preprocessing.label_utils  import load_all_labels
from src.preprocessing.dataset_full import ECGDatasetFull

DATA_PATH    = CFG['data']['path']
RESULTS      = CFG['paths']['results']
FIGURES      = CFG['paths']['figures']
SUPERCLASSES = CFG['data']['superclasses']
LEAD_NAMES   = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(FIGURES, exist_ok=True)
os.makedirs(os.path.join(RESULTS, 'explainability'), exist_ok=True)

# Utility: print text safely on Windows cp1252 terminal
def safe_print(text):
    try:
        print(text)
    except UnicodeEncodeError:
        print(text.encode('ascii', errors='replace').decode('ascii'))

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('GPU: not available -- running on CPU')

In [ ]:
# Read uncertainty_analysis.json to find which base model was used
analysis_path = os.path.join(RESULTS, 'uncertainty_model', 'uncertainty_analysis.json')
assert os.path.exists(analysis_path), (
    f'Missing {analysis_path} -- run Notebook 07 first.'
)
analysis = json.load(open(analysis_path))
BASE_NAME  = analysis['base_model_name']
BASE_AUC   = analysis['base_model_auc']
UNC_AUC    = analysis['uncertainty_model_auc']

print(f'Base model : {BASE_NAME}  (AUC {BASE_AUC:.4f})')
print(f'Uncertainty model AUC : {UNC_AUC:.4f}')
print()

# Loader map (mirrors Notebook 07)
LOADER_MAP = {
    'hubert_ecg_blocks8':   (lambda: load_hubert_blocks(n=8),                 CFG['model']['hubert_hidden_dim']),
    'hubert_ecg_dora_r8':   (lambda: load_hubert_peft(rank=8, use_dora=True), CFG['model']['hubert_hidden_dim']),
    'hubert_ecg_lora_r8':   (lambda: load_hubert_peft(rank=8, use_dora=False),CFG['model']['hubert_hidden_dim']),
    'leadwise_transformer': (lambda: load_leadwise(),                          CFG['model']['leadwise']['d_model']),
}

print('Loading base model (offline)...')
loader_fn, hidden_dim = LOADER_MAP[BASE_NAME]
base_model, _         = loader_fn()

# Wrap and load trained uncertainty weights
uncertainty_model = AleatoricWrapper(base_model, hidden_dim=hidden_dim)
ckpt_path = os.path.join(RESULTS, 'uncertainty_model', 'best_adapter', 'checkpoint.pt')
uncertainty_model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
uncertainty_model.eval()
print(f'Uncertainty model loaded from {ckpt_path}')

In [ ]:
# Load test set and pick 5 representative cases using the pipeline
Y       = load_all_labels(DATA_PATH + 'ptbxl_database.csv',
                          DATA_PATH + 'scp_statements.csv')
test_df = Y[Y.strat_fold == 10]
test_ds = ECGDatasetFull(test_df, DATA_PATH)

pipeline = ECGInferencePipeline(uncertainty_model.to(device), has_uncertainty=True)

# Quick scan of first 200 test records to find interesting cases
N_SCAN = min(200, len(test_ds))
print(f'Scanning {N_SCAN} test records...')

scan_results = []
for i in range(N_SCAN):
    x, y_true = test_ds[i]
    result     = pipeline.predict(x.numpy())
    true_cls   = [SUPERCLASSES[j] for j, v in enumerate(y_true) if v == 1]
    correct    = any(c in result['predicted_classes'] for c in true_cls)
    scan_results.append({
        'idx':         i,
        'uncertainty': result['uncertainty'],
        'confidence':  result['confidence_score'],
        'correct':     correct,
        'true_cls':    true_cls,
        'pred_cls':    result['predicted_classes'],
        'probs':       result['class_probabilities'],
        'unc_level':   result['uncertainty_level'],
    })

df_scan  = pd.DataFrame(scan_results)
_corr    = df_scan[df_scan['correct'] == True]
_wrong   = df_scan[df_scan['correct'] == False]
_multi   = df_scan[df_scan['true_cls'].apply(len) > 1]

case_indices = {
    'Correct + Low Uncertainty':  _corr.nsmallest(1, 'uncertainty')['idx'].iloc[0],
    'Correct + High Uncertainty': _corr.nlargest(1, 'uncertainty')['idx'].iloc[0],
    'Wrong + High Uncertainty':   (_wrong.nlargest(1, 'uncertainty')['idx'].iloc[0]
                                   if len(_wrong) > 0 else 0),
    'Wrong + Low Uncertainty':    (_wrong.nsmallest(1, 'uncertainty')['idx'].iloc[0]
                                   if len(_wrong) > 0 else 1),
    'Multi-label':                (_multi['idx'].iloc[0] if len(_multi) > 0 else 2),
}

print('Selected cases:')
for label, idx in case_indices.items():
    row = df_scan.loc[idx]
    print(f'  [{label}]  idx={idx}  '
          f'true={row["true_cls"]}  pred={row["pred_cls"]}  '
          f'conf={row["confidence"]:.2f}  unc={row["uncertainty"]:.4f}')

In [ ]:
def compute_saliency(model, x_np, target_class_idx, dev):
    """
    Gradient-based input saliency.
    x_np : (12, 1000) numpy array
    Returns (12, 1000) saliency (absolute gradient magnitude).
    Falls back to uniform if gradient computation fails.
    """
    model.eval()
    x_t = (torch.tensor(x_np, dtype=torch.float32)
               .unsqueeze(0)         # (1, 12, 1000)
               .to(dev)
               .requires_grad_(True))
    try:
        mean, _ = model(x_t)
        mean[0, target_class_idx].backward()
        sal = x_t.grad[0].abs().cpu().numpy()  # (12, 1000)
        return sal
    except Exception as e:
        print(f'  Saliency fallback (uniform): {e}')
        return np.ones((12, 1000), dtype=np.float32)


def plot_ecg_with_saliency(signal_np, saliency, case_label, pred_info, save_path=None):
    """
    3x4 grid: 12-lead ECG coloured by saliency on a twin axis.
    signal_np : (12, 1000)
    saliency  : (12, 1000)
    """
    t   = np.linspace(0, 10, 1000)
    sal = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)

    fig, axes = plt.subplots(3, 4, figsize=(16, 8), sharex=True)
    axes = axes.flatten()

    for i, lead in enumerate(LEAD_NAMES):
        ax  = axes[i]
        ax2 = ax.twinx()
        # Saliency as red fill on twin axis (behind signal)
        ax2.fill_between(t, 0, sal[i], color='red', alpha=0.25)
        ax2.set_ylim(0, 1)
        ax2.set_yticks([])
        # ECG signal
        ax.plot(t, signal_np[i], color='black', linewidth=0.7)
        ax.set_title(lead, fontsize=9)
        ax.set_ylabel('mV', fontsize=7)
        ax.grid(True, alpha=0.2)
        ax.set_zorder(ax2.get_zorder() + 1)
        ax.patch.set_visible(False)

    for ax in axes[8:]:
        ax.set_xlabel('Time (s)', fontsize=8)

    fig.suptitle(
        f'{case_label}  |  '
        f'True: {", ".join(pred_info["true_cls"])}  '
        f'Pred: {", ".join(pred_info["pred_cls"])}  '
        f'Conf: {pred_info["confidence"]:.2f}  '
        f'Unc: {pred_info["uncertainty"]:.4f}',
        fontsize=11
    )
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()


# Compute and plot saliency for all 5 cases
case_saliencies = {}
for case_label, idx in case_indices.items():
    row      = df_scan.loc[idx]
    x, _     = test_ds[idx]
    x_np     = x.numpy()   # (12, 1000)

    # Target: the highest-probability predicted class
    pred_probs  = [row['probs'][c] for c in SUPERCLASSES]
    target_idx  = int(np.argmax(pred_probs))

    saliency = compute_saliency(uncertainty_model, x_np, target_idx, device)
    case_saliencies[case_label] = saliency

    # Top-3 most salient leads
    lead_sal = saliency.mean(axis=1)  # (12,) mean over time
    top_leads = [LEAD_NAMES[j] for j in np.argsort(lead_sal)[::-1][:3]]

    pred_info = {
        'true_cls':   row['true_cls'],
        'pred_cls':   row['pred_cls'],
        'confidence': row['confidence'],
        'uncertainty':row['uncertainty'],
    }
    fig_path = os.path.join(FIGURES, f'saliency_{case_label.replace(" ", "_").replace("+","plus")}.png')
    plot_ecg_with_saliency(x_np, saliency, case_label, pred_info, save_path=fig_path)
    print(f'  Case: {case_label}')
    print(f'    Most salient leads : {top_leads}')

In [ ]:
# Free ECG model from GPU before loading Qwen3
uncertainty_model.cpu()
torch.cuda.empty_cache()

# Switch to online mode so Qwen3 can be downloaded on first run
# (subsequent runs use the local HuggingFace cache)
os.environ['TRANSFORMERS_OFFLINE'] = '0'

from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN_MODEL = 'Qwen/Qwen3-0.6B'   # swap to Qwen3-1.7B or Qwen3-4B for richer output

print(f'Loading {QWEN_MODEL} ...')
print('(Downloads ~400 MB on first run, then cached)')

qwen_tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL, trust_remote_code=True
)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map='auto',
)
qwen_model.eval()
print(f'Qwen3 loaded  |  device: {next(qwen_model.parameters()).device}')

In [ ]:
SYSTEM_PROMPT = (
    'You are a clinical AI assistant helping cardiologists interpret ECG predictions '
    'from a deep learning model. Be concise (3-5 sentences), clinically grounded, '
    'and always remind the reader that AI results require clinical correlation. '
    'Do not diagnose -- interpret what the model found.'
)


def build_prompt(row, saliency):
    """Format structured model output into a Qwen3 user prompt."""
    prob_lines = '  '.join(
        f"{c}:{row['probs'][c]:.2f}" for c in SUPERCLASSES
    )
    lead_sal   = saliency.mean(axis=1)
    top_leads  = ', '.join([LEAD_NAMES[j] for j in np.argsort(lead_sal)[::-1][:3]])

    return (
        f"ECG ANALYSIS RESULT\n"
        f"True diagnosis (if known): {', '.join(row['true_cls']) or 'unknown'}\n"
        f"Predicted class:           {', '.join(row['pred_cls'])}\n"
        f"Confidence score:          {row['confidence']:.2f}\n"
        f"Class probabilities:       {prob_lines}\n"
        f"Signal uncertainty:        {row['uncertainty']:.4f} ({row['unc_level']})\n"
        f"Top salient leads:         {top_leads}\n\n"
        f"Please provide a 3-5 sentence clinical interpretation of these results."
    )


def generate_explanation(prompt_text, max_new_tokens=220, temperature=0.3):
    """Call Qwen3 to generate a clinical explanation from the structured prompt."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt_text},
    ]
    # enable_thinking=False: skip chain-of-thought for concise clinical output
    try:
        text = qwen_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        text = qwen_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

    inputs = qwen_tokenizer(text, return_tensors='pt').to(next(qwen_model.parameters()).device)

    with torch.no_grad():
        out = qwen_model.generate(
            **inputs,
            max_new_tokens   = max_new_tokens,
            temperature      = temperature,
            do_sample        = True,
            pad_token_id     = qwen_tokenizer.eos_token_id,
        )

    new_tokens  = out[0][inputs.input_ids.shape[1]:]
    raw_text    = qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    # Strip any remaining <think>...</think> blocks if thinking mode leaked
    if '<think>' in raw_text and '</think>' in raw_text:
        raw_text = raw_text[raw_text.index('</think>') + len('</think>'):].strip()
    return raw_text


# Generate explanations for all 5 cases
explanations = {}
for case_label, idx in case_indices.items():
    row     = df_scan.loc[idx]
    saliency = case_saliencies[case_label]
    prompt  = build_prompt(row, saliency)

    t0  = time.perf_counter()
    exp = generate_explanation(prompt)
    dt  = time.perf_counter() - t0

    explanations[case_label] = exp
    print(f'\n[{case_label}]  ({dt:.1f}s)')
    safe_print(exp)
    print()

In [ ]:
# Display all 5 cases: structured results + Qwen3 explanation per case
SEP = '=' * 70
for case_label, idx in case_indices.items():
    row   = df_scan.loc[idx]
    probs = row['probs']

    print(SEP)
    print(f'CASE: {case_label}')
    print(SEP)
    print(f'  True diagnosis  : {(chr(44)+" ").join(row["true_cls"]) or "None"}')
    print(f'  Predicted class : {(chr(44)+" ").join(row["pred_cls"])}')
    print(f'  Probabilities   : '
          + '  '.join(f'{c}:{probs[c]:.3f}' for c in SUPERCLASSES))
    print(f'  Confidence      : {row["confidence"]:.3f}')
    print(f'  Uncertainty     : {row["uncertainty"]:.4f}  ({row["unc_level"]})')
    print(f'  Correct         : {"Yes" if row["correct"] else "No"}')
    print()
    print('  -- Qwen3 Explanation --')
    exp_lines = explanations.get(case_label, '(not generated)')
    for line in exp_lines.splitlines():
        safe_print(f'  {line}')
    print()

In [ ]:
# Build and save the full explanation report
report = []
for case_label, idx in case_indices.items():
    row      = df_scan.loc[idx]
    saliency = case_saliencies[case_label]
    lead_sal = saliency.mean(axis=1)
    top_leads = [LEAD_NAMES[j] for j in np.argsort(lead_sal)[::-1][:3]]

    report.append({
        'case_type':          case_label,
        'test_set_idx':       int(idx),
        'true_classes':       row['true_cls'],
        'predicted_classes':  row['pred_cls'],
        'class_probabilities':row['probs'],
        'confidence':         float(row['confidence']),
        'uncertainty':        float(row['uncertainty']),
        'uncertainty_level':  row['unc_level'],
        'correct':            bool(row['correct']),
        'top_salient_leads':  top_leads,
        'qwen3_explanation':  explanations.get(case_label, ''),
        'qwen3_model':        QWEN_MODEL,
    })

out_path = os.path.join(RESULTS, 'explainability', 'explanations.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump({'base_model': BASE_NAME, 'cases': report}, f, indent=2, ensure_ascii=False)

print(f'Saved -> {out_path}')
print(f'{len(report)} cases recorded.')